# Optuna Hyperparameter Tuning

Run Optuna studies for TGNN-Solv and supported baselines.

CLI equivalent: `python scripts/experiments/run_optuna.py --models tgnn_solv,direct_gnn --n-trials 20`
See also `docs/training.md` and `docs/script_reference.md`.

The maintained `split_late` backbone comparison is a fixed-config study via
`configs/paper_config_split_late.yaml`, not the default Optuna target.


## The optimization view of Optuna

Hyperparameter tuning here is formulated as a search problem

$$
\theta^* = \arg\min_{\theta \in \Theta} \mathcal{J}(\theta),
$$

where in the standard single-objective setting

$$
\mathcal{J}(\theta) = \mathrm{MAE}_{\mathrm{val}}(\theta)
$$

or a closely related validation metric after training a model with parameters \(\theta\).

The trial history forms the sequence

$$
\mathcal{H}_t = \left\{\left(\theta^{(j)}, \mathcal{J}(\theta^{(j)})\right)\right\}_{j=1}^{t},
$$

and the next trial is sampled conditionally on that history:

$$
\theta^{(t+1)} \sim q\!\left(\theta \mid \mathcal{H}_t\right).
$$

In practice, this means the notebook is not looking for the “best architecture in the abstract”,
but for the best hyperparameter setting relative to a specific validation split and a specific training budget.


## Step 1. Initialize the environment and the search space

The notebook first constructs `OptunaTuner` and fixes the processed split and the model set to tune.
This is the place to be especially careful not to mix debug-scale tuning budgets with real comparison budgets.


In [ ]:
from pathlib import Path
import sys
import json

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"
TABLES_DIR = PROJECT_ROOT / "tables"
NOTEBOOK_FIG_DIR = FIGURES_DIR / "notebooks"
NOTEBOOK_RESULTS_DIR = RESULTS_DIR / "notebooks"
NOTEBOOK_FIG_DIR.mkdir(parents=True, exist_ok=True)
NOTEBOOK_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import torch
from tgnn_solv.optuna_tuner import OptunaTuner, AVAILABLE_MODELS

print(f"Project root: {PROJECT_ROOT}")
print(f"Available models: {sorted(AVAILABLE_MODELS)}")


## Step 2. Configure one study

The next block defines a concrete study for the chosen model. This is the right moment to verify
the objective, the number of trials, and the hyperparameter ranges before running the search.


In [ ]:
processed_dir = PROJECT_ROOT / "notebooks" / "data" / "processed"
train_df, val_df, test_df = OptunaTuner.load_csv_splits(
    str(processed_dir / "train.csv"),
    str(processed_dir / "val.csv"),
    str(processed_dir / "test.csv"),
)
datasets = OptunaTuner.build_datasets(train_df, val_df, test_df, cache=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


In [ ]:
tuner = OptunaTuner(
    datasets=datasets,
    device=device,
    seed=42,
    num_workers=0,
    tune_arch=True,
)


## Step 3. Analyze one completed search

After the study finishes, it is important to inspect not only the best trial, but the search history as a whole:
how consistently the sampler moved toward good regions, and whether the trial budget looks large enough to support
a stable conclusion.


In [ ]:
study = tuner.run_study("tgnn_solv", n_trials=10)
study.best_value, study.best_params


## Step 4. Compare multiple search spaces

The final block is useful when you want to test whether one model's advantage could be explained simply by a more
favorable search space. At this point tuning stops being only an engineering procedure and becomes part of a fair
research comparison.


In [ ]:
models = ["tgnn_solv", "direct_gnn"]
results = {}
for name in models:
    study = tuner.run_study(name, n_trials=5, study_name=f"demo_{name}")
    results[name] = {"best_value": study.best_value, "best_params": study.best_params}

output_path = NOTEBOOK_RESULTS_DIR / "optuna_demo_results.json"
output_path.write_text(json.dumps(results, indent=2), encoding="utf-8")
print(f"Saved notebook summary to {output_path}")
results
